In [148]:
import pandas as pd
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import stopwords
import string
import nltk
from textblob import TextBlob
from contractions import contractions_dict
import numpy as np

In [105]:
reviews = pd.read_csv('../Data/raw/20190928-reviews.csv')
items = pd.read_csv('../Data/raw/20190928-items.csv')

In [106]:
reviews.head()

,asin,name,rating,date,verified,title,body,helpfulVotes
0,B0000SX2UC,Janet,3,"October 11, 2005",False,"Def not best, but not worst",I had the Samsung A600 for awhile which is abs...,1.0
1,B0000SX2UC,Luke Wyatt,1,"January 7, 2004",False,Text Messaging Doesn't Work,Due to a software issue between Nokia and Spri...,17.0
2,B0000SX2UC,Brooke,5,"December 30, 2003",False,Love This Phone,"This is a great, reliable phone. I also purcha...",5.0
3,B0000SX2UC,amy m. teague,3,"March 18, 2004",False,"Love the Phone, BUT...!","I love the phone and all, because I really did...",1.0
4,B0000SX2UC,tristazbimmer,4,"August 28, 2005",False,"Great phone service and options, lousy case!",The phone has been great for every purpose it ...,1.0


In [107]:
reviews = reviews[['asin','verified','title','body']]

In [108]:
items.head()

,asin,brand,title,url,image,rating,reviewUrl,totalReviews,prices
0,B0000SX2UC,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...,https://www.amazon.com/Dual-Band-Tri-Mode-Acti...,https://m.media-amazon.com/images/I/2143EBQ210...,3.0,https://www.amazon.com/product-reviews/B0000SX2UC,14,NaN
1,B0009N5L7K,Motorola,Motorola I265 phone,https://www.amazon.com/Motorola-i265-I265-phon...,https://m.media-amazon.com/images/I/419WBAVDAR...,2.9,https://www.amazon.com/product-reviews/B0009N5L7K,7,$49.95
2,B000SKTZ0S,Motorola,MOTOROLA C168i AT&T CINGULAR PREPAID GOPHONE C...,https://www.amazon.com/MOTOROLA-C168i-CINGULAR...,https://m.media-amazon.com/images/I/71b+q3ydkI...,2.6,https://www.amazon.com/product-reviews/B000SKTZ0S,22,NaN
3,B00198M12M,Nokia,Nokia 6500 Slide Black/silver Unlocked Cell Phone,https://www.amazon.com/Nokia-6500-Slide-silver...,https://m.media-amazon.com/images/I/41ss4HpLkL...,2.4,https://www.amazon.com/product-reviews/B00198M12M,5,NaN
4,B001AO4OUC,Motorola,Motorola i335 Cell Phone Boost Mobile,https://www.amazon.com/Motorola-i335-Phone-Boo...,https://m.media-amazon.com/images/I/710UO8gdT+...,3.3,https://www.amazon.com/product-reviews/B001AO4OUC,21,NaN


In [109]:
items = items[['asin','brand','title']]

In [110]:
reviews = pd.merge(reviews, items, how="left", left_on="asin", right_on="asin")

In [111]:
reviews.rename(columns={"rating_x": "rating", "title_x": "title", "title_y": "name_pro", "rating_y": "overall_rating"}, inplace=True)


In [112]:
reviews.drop(columns=['asin'], inplace=True)


In [113]:
reviews.head()

,verified,title,body,brand,name_pro
0,False,"Def not best, but not worst",I had the Samsung A600 for awhile which is abs...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
1,False,Text Messaging Doesn't Work,Due to a software issue between Nokia and Spri...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
2,False,Love This Phone,"This is a great, reliable phone. I also purcha...",Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
3,False,"Love the Phone, BUT...!","I love the phone and all, because I really did...",Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
4,False,"Great phone service and options, lousy case!",The phone has been great for every purpose it ...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...


In [114]:
reviews.to_csv("../Data/processed/merged_reviews.csv", index=False, encoding='utf-8')

In [115]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82815 entries, 0 to 82814
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   verified  82815 non-null  bool  
 1   title     82789 non-null  object
 2   body      82789 non-null  object
 3   brand     82815 non-null  object
 4   name_pro  82815 non-null  object
dtypes: bool(1), object(4)
memory usage: 2.6+ MB


In [116]:
reviews.isnull().sum()

verified     0
title       26
body        26
brand        0
name_pro     0
dtype: int64

In [117]:
reviews.dropna(inplace=True)
reviews.isnull().sum()

verified    0
title       0
body        0
brand       0
name_pro    0
dtype: int64

In [118]:
print(reviews['body'].unique()[:10])

["I had the Samsung A600 for awhile which is absolute doo doo. You can read my review on it and detect my rage at the stupid thing. It finally died on me so I used this Nokia phone I bought in a garage sale for $1. I wonder y she sold it so cheap?... Bad: ===> I hate the menu. It takes forever to get to what you want because you have to scroll endlessly. Usually phones have numbered categories so u can simply press the # and get where you want to go. ===> It's a pain to put it on silent or vibrate. If you're in class and it rings, you have to turn it off immediately. There's no fast way to silence the damn thing. Always remember to put it on silent! I learned that the hard way. ===> It's so true about the case. It's a mission to get off and will break ur nails in the process. Also, you'll damage the case each time u try. For some reason the phone started giving me problems once I did succeed in opening it. ===> Buttons could be a bit bigger. Vibration could be stronger. Good: ===> Rece

In [119]:
import langdetect

def detect_language(text):
    try:
        return langdetect.detect(text)
    except:
        return 'unknown'

reviews['language'] = reviews['body'].apply(detect_language)

reviews = reviews[reviews['language'] != 'es']

reviews.drop('language', axis=1, inplace=True)

In [ ]:
reviews.to_csv('../Data/processed/langdetected.csv', index=False, encoding='utf-8')

In [151]:
reviews = pd.read_csv('../Data/processed/langdetected.csv')

In [152]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
stop = set(stopwords.words('english'))
punc = set(string.punctuation)
lemma = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ducdu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ducdu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ducdu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [153]:
reviews.head(5)

,verified,title,body,brand,name_pro
0,False,"Def not best, but not worst",I had the Samsung A600 for awhile which is abs...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
1,False,Text Messaging Doesn't Work,Due to a software issue between Nokia and Spri...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
2,False,Love This Phone,"This is a great, reliable phone. I also purcha...",Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
3,False,"Love the Phone, BUT...!","I love the phone and all, because I really did...",Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...
4,False,"Great phone service and options, lousy case!",The phone has been great for every purpose it ...,Nokia,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...


In [154]:
import re
contractions_dict = { "ain't": "are not","'s":" is","aren't": "are not",
                     "can't": "cannot","can't've": "cannot have",
                     "'cause": "because","could've": "could have","couldn't": "could not",
                     "couldn't've": "could not have", "didn't": "did not","doesn't": "does not",
                     "Doesn't": "Does not", "Didn't": "Did not",
                     "don't": "do not","hadn't": "had not","hadn't've": "had not have",
                     "hasn't": "has not","haven't": "have not","he'd": "he would",
                     "he'd've": "he would have","he'll": "he will", "he'll've": "he will have",
                     "how'd": "how did","how'd'y": "how do you","how'll": "how will",
                     "I'd": "I would", "I'd've": "I would have","I'll": "I will",
                     "I'll've": "I will have","I'm": "I am","I've": "I have", "isn't": "is not",
                     "it'd": "it would","it'd've": "it would have","it'll": "it will",
                     "it'll've": "it will have", "let's": "let us","ma'am": "madam",
                     "mayn't": "may not","might've": "might have","mightn't": "might not",
                     "mightn't've": "might not have","must've": "must have","mustn't": "must not",
                     "mustn't've": "must not have", "needn't": "need not",
                     "needn't've": "need not have","o'clock": "of the clock","oughtn't": "ought not",
                     "oughtn't've": "ought not have","shan't": "shall not","sha'n't": "shall not",
                     "shan't've": "shall not have","she'd": "she would","she'd've": "she would have",
                     "she'll": "she will", "she'll've": "she will have","should've": "should have",
                     "shouldn't": "should not", "shouldn't've": "should not have","so've": "so have",
                     "that'd": "that would","that'd've": "that would have", "there'd": "there would",
                     "there'd've": "there would have", "they'd": "they would",
                     "they'd've": "they would have","they'll": "they will",
                     "they'll've": "they will have", "they're": "they are","they've": "they have",
                     "to've": "to have","wasn't": "was not","we'd": "we would",
                     "we'd've": "we would have","we'll": "we will","we'll've": "we will have",
                     "we're": "we are","we've": "we have", "weren't": "were not","what'll": "what will",
                     "what'll've": "what will have","what're": "what are", "what've": "what have",
                     "when've": "when have","where'd": "where did", "where've": "where have",
                     "who'll": "who will","who'll've": "who will have","who've": "who have",
                     "why've": "why have","will've": "will have","won't": "will not",
                     "won't've": "will not have", "would've": "would have","wouldn't": "would not",
                     "wouldn't've": "would not have","y'all": "you all", "y'all'd": "you all would",
                     "y'all'd've": "you all would have","y'all're": "you all are",
                     "y'all've": "you all have", "you'd": "you would","you'd've": "you would have",
                     "you'll": "you will","you'll've": "you will have", "you're": "you are",
                     "you've": "you have"}

contractions_re=re.compile('(%s)' % '|'.join(contractions_dict.keys()))

def expand_contractions(text,contractions_dict=contractions_dict):
  def replace(match):
    return contractions_dict[match.group(0)]
  return contractions_re.sub(replace, text)

reviews['body']=reviews['body'].apply(lambda x:expand_contractions(x))
reviews['title']=reviews['title'].apply(lambda x:expand_contractions(x))

In [155]:
reviews.isnull().sum()

verified    0
title       0
body        0
brand       0
name_pro    0
dtype: int64

In [156]:
keywords = reviews["brand"].apply(lambda x: x.lower()).unique().tolist()
keywords.append("phone")

def clean_text(text):
    text = text.lower()
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(' +', ' ', text)
    wordList = text.split()
    wordList = [word for word in wordList if word not in stop]
    wordList = [word for word in wordList if word not in keywords]
    wordList = [lemma.lemmatize(word) for word in wordList]
    return " ".join(wordList)

def clean_title(text):
    text = text.lower()
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(' +', ' ', text)
    wordList = text.split()
    wordList = [lemma.lemmatize(word) for word in wordList]
    return " ".join(wordList)

reviews["clean_text"] = reviews["body"].astype("str").apply(clean_text)
reviews["clean_title"] = reviews["title"].astype("str").apply(clean_title)

In [157]:
print(reviews['clean_text'].unique()[:3])

['awhile absolute doo doo read review detect rage stupid thing finally died used bought garage sale wonder sold cheap bad hate menu take forever get want scroll endlessly usually phone numbered category u simply press get want go pain put silent vibrate class ring turn immediately fast way silence damn thing always remember put silent learned hard way true case mission get break ur nail process also damage case time u try reason started giving problem succeed opening button could bit bigger vibration could stronger good reception shabby using elevator remarkable feat considering old would lose service simply putting pocket compared old work quite well ring tone loud enough hear actually charge quickly great battery life heat like potatoe oven either long convos nice bright large screen cute way customize scroll bar set purple pink aqua orange etc overall okay serf purpose definitely pale comparison new phone coming sprint get get great'
 'due software issue sprint text messaging capabi

In [158]:
reviews = reviews[
    (reviews['clean_text'].str.strip() != '') &
    (reviews['clean_title'].str.strip() != '')
]

In [159]:
reviews.to_csv('../Data/processed/reviews_clean.csv', index=False, encoding='utf-8')